In [101]:
import torch
import torch.nn as nn
import numpy as np


In [102]:
# We are doing this to teach a model how to predict the next letter in a sequence based on what it has seen before.

In [103]:
text = "pytorch or keras" # The text on which we will do text generation
chars = sorted(list(set(text))) # Basically we want a non duplicated sorted list and this is the only option for that 

In [104]:
# Examine chars : 
chars # See how the list contais no duplicates of o and is sroted too

[' ', 'a', 'c', 'e', 'h', 'k', 'o', 'p', 'r', 's', 't', 'y']

In [105]:
# The len of our vocabulary
vocab_size = len(chars)

In [106]:
# Now we create two loookup tables 
char_to_ix = {ch: i for i, ch in enumerate(chars)} # Given a character returns it's index.
ix_to_char = {i: ch for i, ch in enumerate(chars)} # Given an integer return the character assigned to it.

In [107]:
char_to_ix

{' ': 0,
 'a': 1,
 'c': 2,
 'e': 3,
 'h': 4,
 'k': 5,
 'o': 6,
 'p': 7,
 'r': 8,
 's': 9,
 't': 10,
 'y': 11}

In [108]:
# When data goes into the model, we use char_to_ix to translate letters to numbers.
# When guesses come out of the model, we use ix_to_char to translate those numbers back into human-readable letters

In [109]:
# Now we define the inputs and labels from our text
inputs = [char_to_ix[ch] for ch in text[:-1]]   # "pytorc"
targets = [char_to_ix[ch] for ch in text[1:]]   # "ytorch"

In [110]:
# Why do we this ? To create pairs from which we will predict 
# There will be len(text)*2 pairs

Why shift them by one letter? Because the model processes data sequentially, one step at a time. By lining them up side-by-side, we create len(text) perfect training pairs (Assume our text is "cat") : 

| Time Step | Model Input | Correct Target (Next Letter) |
| :--- | :--- | :--- |
| 1 | 0 ( 'c' ) | 1 ( 'a' ) |
| 2 | 1 ( 'a' ) | 2 ( 't' ) |

When the model is at Time Step 1, it looks at `'c'` and tries to guess what comes next. We check its guess against the target `'a'`.

Then, it moves to Time Step 2, looks at `'a'`, and tries to guess the next letter. We check its guess against the target `'t'`.


In [111]:
# This way we are trying to predict the next word for each input

In [112]:
# Now what we do is OneHotEndcoding and by doing this we convert every single character to a vector 
# One-hot encoding converts every single integer index into a vector (a row of numbers) where every value is 0.0,
# except for a single 1.0 at the index position

# Index 3 ('p') becomes: [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0] (The 1.0 is at position 3)

In [113]:
# For this conversion we use this functionn to create vectors
def to_one_hot(indices, vocab_size):
    # 1. Create a matrix of zeros. Rows = number of letters, Columns = vocabulary size (7)
    one_hot = np.zeros((len(indices), vocab_size), dtype=np.float32)
    
    # 2. Loop through each index and plant a '1.0' in the correct column
    for i, idx in enumerate(indices):
        one_hot[i][idx] = 1.0
        
    # 3. Turn the NumPy array into a PyTorch tensor
    return torch.tensor(one_hot)

In [114]:
input_tensors = to_one_hot(inputs, vocab_size).unsqueeze(1)

In [115]:
input_tensors # If we correlate this with our text : "pytorch or keras" we can understand what's going on 

tensor([[[0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.]],

        [[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.]],

        [[0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.]],

        [[0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.]],

        [[1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.]],

        [[1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.]],

        [[0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.]],

        [[0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]]])

In [116]:
target_tensors = torch.tensor(targets, dtype=torch.long)
#  PyTorch's loss function (CrossEntropyLoss) is specifically designed to accept raw integers as targets. 
# It handles the comparison math under the hood automatically, saving memory and keeping our target code simple.

In [117]:
target_tensors # The indices of next word

tensor([11, 10,  6,  8,  2,  4,  0,  6,  8,  0,  5,  3,  8,  1,  9])

In [118]:
# Creating our simple RNN model : 
class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleRNN, self).__init__()
        self.hidden_size = hidden_size
        self.rnn = nn.RNN(input_size, hidden_size, num_layers=1) # Create the RNN layer where we have 12 inputs and 16 output to HL
        self.fc = nn.Linear(hidden_size, output_size) # Here we do the exact opposite of RNN layer : 16 I/P and 12 O/P

    # An RNN has memory because of the Hidden State
    def init_hidden(self):
        return torch.zeros(1, 1, self.hidden_size) # Shape: [1, 1, 16]
    
    # Now moving or to forward propogation
    def forward(self, x, hidden):
        # x shape: [seq_len, batch_size, input_size] -> 
        out, hidden = self.rnn(x, hidden)
        
        # out shape: [seq_len, batch_size, hidden_size] -> 
        out = self.fc(out.squeeze(1)) # Collapse batch dim for Linear layer
        
        return out, hidden

In [119]:
# Defining the arguments to be passed
input_size = vocab_size    # 12
hidden_size = 16           # 16
output_size = vocab_size   # 12

In [120]:
# Instantiate the model
model = SimpleRNN(input_size, hidden_size, output_size)

In [121]:
# Choosen loss function and optimizer for the model
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

In [122]:
# The training loop
epochs = 100
for epoch in range(epochs):
    hidden = model.init_hidden() # 1. Clear memory
    optimizer.zero_grad()        # 2. Reset calculations
    
    output, hidden = model(input_tensors, hidden) # 3. Forward pass
    loss = criterion(output, target_tensors)       # 4. Score guesses
    
    loss.backward()              # 5. Trace mistakes
    optimizer.step()             # 6. Adjust weight dials

    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

Epoch [20/100], Loss: 0.0091
Epoch [40/100], Loss: 0.0010
Epoch [60/100], Loss: 0.0005
Epoch [80/100], Loss: 0.0004
Epoch [100/100], Loss: 0.0004


In [123]:
# Now testing / Inferencing 

print("\nTesting the model:")
model.eval()
with torch.no_grad():
    hidden = model.init_hidden()
    current_char = "p"
    result = current_char
    
    for _ in range(len(text) - 1):
        idx = char_to_ix[current_char]
        input_tensor = to_one_hot([idx], vocab_size).unsqueeze(1)
        
        output, hidden = model(input_tensor, hidden)
        pred_idx = torch.argmax(output, dim=1).item()
        
        current_char = ix_to_char[pred_idx]
        result += current_char

    print(f"Input: 'p' -> Predicted Sequence: '{result}'")




Testing the model:
Input: 'p' -> Predicted Sequence: 'pytorch or keras'
